In [8]:
class Anotacao:
    def __init__ (self, pagina, paragrafo, texto):
        self.pagina = pagina
        self.paragrafo = paragrafo
        self.texto = texto
        # Usando tupla como chave: o Python compara tuplas elemento a elemento.
        # Ou seja, primeiro ordenará por página e, em caso de empate, por parágrafo.
        self.chave = (pagina, paragrafo) 

    # Adicionado para facilitar os testes. Define como o objeto será impresso na tela.
    def __repr__(self):
        return f"Anotacao(Pg:{self.pagina}, Par:{self.paragrafo})"

class Node:
    def __init__(self, t, leaf=True):
        self.chaves = [] # Armazena os objetos Anotacao ordenados
        self.filhos = [] # Armazena referências para outros objetos Node
        # t é o grau mínimo. Define as regras de capacidade do nó:
        # Mínimo de chaves por nó (exceto raiz) = t - 1
        # Máximo de chaves por nó = 2t - 1
        self.t = t 
        self.leaf = leaf # Verdadeiro se o nó não tem filhos (está na base da árvore)

class ArvoreB:
    def __init__ (self, t):
        self.raiz = Node(t, leaf=True) # A árvore sempre nasce com uma raiz vazia que é folha
        self.t = t

    def inserir(self, anotacao):
        # O método inserir é a "porta de entrada". Ele só se preocupa com uma coisa:
        # "A raiz está cheia?" Se sim, a árvore precisa crescer para cima.
        if len(self.raiz.chaves) == (2 * self.t) - 1:
            # Cenário A: Raiz Cheia!
            nova_raiz = Node(self.t, leaf=False) 
            nova_raiz.filhos.append(self.raiz) # A raiz antiga vira o primeiro filho da nova raiz
            self.raiz = nova_raiz  
            # Divide a raiz antiga (que agora é filha) em duas partes e sobe a chave do meio
            self._dividir_filho(nova_raiz, 0, nova_raiz.filhos[0]) 
            # Agora que há espaço no topo, desce inserindo
            self._inserir_nao_cheio(nova_raiz, anotacao) 
        else: 
            # Cenário B: Raiz tem espaço, basta iniciar a descida
            self._inserir_nao_cheio(self.raiz, anotacao)

    def _dividir_filho(self, pai, i, filho):
        # Este método pega um filho que está na capacidade máxima (2t - 1 chaves)
        # e o divide no meio. Cria um novo nó para a metade direita.
        novo_node = Node(self.t, leaf=filho.leaf) 
        
        # O novo nó recebe a metade direita das chaves (de 't' até o fim)
        novo_node.chaves = filho.chaves[self.t:] 
        
        # A chave do meio (índice t-1) sobe para o nó pai para separar os dois filhos
        pai.chaves.insert(i, filho.chaves[self.t - 1]) 
        
        # O filho original fica apenas com a metade esquerda das chaves
        filho.chaves = filho.chaves[:self.t - 1] 

        # Se não for folha, os filhos do nó dividido também precisam ser distribuídos
        if not filho.leaf:
            novo_node.filhos = filho.filhos[self.t:] 
            filho.filhos = filho.filhos[:self.t] 

        # Conecta o novo nó (metade direita) ao pai, logo após o filho original
        pai.filhos.insert(i + 1, novo_node) 

    def _inserir_nao_cheio(self, no, anotacao):
        # Função recursiva que garante que nunca entrará num nó cheio
        i = len(no.chaves) - 1

        if no.leaf:
            # Chegamos no destino! Abre um espaço (None) no final da lista
            no.chaves.append(None) 
            # Faz um "shift" empurrando as chaves maiores para a direita
            while i >= 0 and anotacao.chave < no.chaves[i].chave:
                no.chaves[i + 1] = no.chaves[i]
                i -= 1
            # Insere a nova anotação no buraco que sobrou
            no.chaves[i + 1] = anotacao
        else:
            # Não é folha, precisamos encontrar por qual filho descer
            while i >= 0 and anotacao.chave < no.chaves[i].chave:
                i -= 1
            i += 1 # i agora aponta para o índice do filho correto
            
            # Checagem preventiva: o filho destino está cheio?
            if len(no.filhos[i].chaves) == (2 * self.t) - 1:
                # Se sim, divide o filho ANTES de descer
                self._dividir_filho(no, i, no.filhos[i])
                
                # Como a divisão subiu uma chave para o nó atual, verificamos 
                # de novo se devemos descer pela metade esquerda ou direita
                if anotacao.chave > no.chaves[i].chave:
                    i += 1
                    
            # Desce recursivamente com a garantia de que o nó destino tem espaço
            self._inserir_nao_cheio(no.filhos[i], anotacao)

    def consultar(self, pagina, paragrafo):
        # Prepara a chave de busca para comparar com as tuplas da árvore
        chave_busca = (pagina, paragrafo)
        return self._consultar_no(self.raiz, chave_busca)

    def _consultar_no(self, no, chave_busca):
        i = 0
        # Caminha pelas chaves do nó atual procurando a posição certa
        while i < len(no.chaves) and chave_busca > no.chaves[i].chave:
            i += 1

        # Achamos a chave exata neste nó!
        if i < len(no.chaves) and no.chaves[i].chave == chave_busca:
            return no.chaves[i] 

        # Chegamos no fundo da árvore e não achamos: a anotação não existe
        if no.leaf:
            return None

        # Recursão para buscar no filho correspondente ao intervalo encontrado
        return self._consultar_no(no.filhos[i], chave_busca)

    def listar_anotacoes(self):
        # Retorna todas as anotações ordenadas (In-Order Traversal)
        resultado = []
        self._listar_no(self.raiz, resultado)
        return resultado

    def _listar_no(self, no, resultado):
        if no is not None:
            for i in range(len(no.chaves)):
                # Visita a subárvore à esquerda da chave i
                if not no.leaf:
                    self._listar_no(no.filhos[i], resultado)
                # Coleta a chave i
                resultado.append(no.chaves[i])
            
            # Por fim, visita a subárvore à direita da ÚLTIMA chave
            if not no.leaf:
                self._listar_no(no.filhos[len(no.chaves)], resultado)

    def remover(self, pagina, paragrafo):
        chave_remover = (pagina, paragrafo)
        if not self.raiz:
            return

        self._remover_do_no(self.raiz, chave_remover)

        # Se a raiz doou sua única chave durante a remoção, ela fica vazia.
        # Nesse caso, o único filho dela assume como nova raiz (diminuindo a altura da árvore)
        if len(self.raiz.chaves) == 0:
            if self.raiz.leaf:
                self.raiz = Node(self.t, leaf=True) 
            else:
                self.raiz = self.raiz.filhos[0] 

    def _remover_do_no(self, no, chave):
        i = 0
        while i < len(no.chaves) and chave > no.chaves[i].chave:
            i += 1

        # A chave a ser removida ESTÁ neste nó
        if i < len(no.chaves) and no.chaves[i].chave == chave:
            if no.leaf:
                # Caso 1: Nó é folha. Simplesmente deleta (remoção segura graças ao _preencher)
                no.chaves.pop(i)
            else:
                # Caso 2: Nó interno. Precisamos de um substituto válido das subárvores
                if len(no.filhos[i].chaves) >= self.t:
                    # Substitui pelo antecessor (maior valor à esquerda)
                    antecessor = self._obter_antecessor(no, i)
                    no.chaves[i] = antecessor
                    self._remover_do_no(no.filhos[i], antecessor.chave)
                elif len(no.filhos[i + 1].chaves) >= self.t:
                    # Substitui pelo sucessor (menor valor à direita)
                    sucessor = self._obter_sucessor(no, i)
                    no.chaves[i] = sucessor
                    self._remover_do_no(no.filhos[i + 1], sucessor.chave)
                else:
                    # Ambos os filhos estão no limite mínimo. Funde os dois nós
                    self._fundir(no, i)
                    self._remover_do_no(no.filhos[i], chave)

        # A chave NÃO ESTÁ neste nó, precisamos descer
        elif not no.leaf:
            # Caso 3: Checagem preventiva. O filho para onde vamos descer tem o mínimo de chaves?
            if len(no.filhos[i].chaves) == self.t - 1:
                self._preencher(no, i) # Engorda o nó pedindo emprestado ou fundindo

            # Se houve fusão, o índice 'i' pode ter mudado
            if i > len(no.chaves) or (i < len(no.chaves) and chave > no.chaves[i].chave):
                i += 1

            self._remover_do_no(no.filhos[i], chave)

    # --- MÉTODOS AUXILIARES DA REMOÇÃO ---

    def _obter_antecessor(self, no, i):
        # Desce o máximo possível à direita na subárvore esquerda
        atual = no.filhos[i]
        while not atual.leaf:
            atual = atual.filhos[-1]
        return atual.chaves[-1]

    def _obter_sucessor(self, no, i):
        # Desce o máximo possível à esquerda na subárvore direita
        atual = no.filhos[i + 1]
        while not atual.leaf:
            atual = atual.filhos[0]
        return atual.chaves[0]

    def _preencher(self, no, i):
        # Garante que um nó não sofra "underflow" (ficar com menos que t-1 chaves)
        if i > 0 and len(no.filhos[i - 1].chaves) >= self.t:
            self._pegar_emprestado_esq(no, i)
        elif i < len(no.filhos) - 1 and len(no.filhos[i + 1].chaves) >= self.t:
            self._pegar_emprestado_dir(no, i)
        else:
            # Se nenhum irmão pode emprestar, junta com um deles
            if i < len(no.filhos) - 1:
                self._fundir(no, i)
            else:
                self._fundir(no, i - 1)

    def _pegar_emprestado_esq(self, no, i):
        # O irmão da esquerda doa sua maior chave para o PAI
        # e o PAI doa sua chave separadora para o filho necessitado
        filho = no.filhos[i]
        irmao = no.filhos[i - 1]

        filho.chaves.insert(0, no.chaves[i - 1])
        if not filho.leaf:
            filho.filhos.insert(0, irmao.filhos.pop())

        no.chaves[i - 1] = irmao.chaves.pop()

    def _pegar_emprestado_dir(self, no, i):
        # O irmão da direita doa sua menor chave para o PAI
        # e o PAI doa sua chave separadora para o filho necessitado
        filho = no.filhos[i]
        irmao = no.filhos[i + 1]

        filho.chaves.append(no.chaves[i])
        if not filho.leaf:
            filho.filhos.append(irmao.filhos.pop(0))

        no.chaves[i] = irmao.chaves.pop(0)

    def _fundir(self, no, i):
        # Junta dois irmãos (i e i+1) em um nó só.
        # A chave separadora do PAI desce para formar o novo nó unificado.
        filho = no.filhos[i]
        irmao = no.filhos[i + 1]

        # Desce a chave do pai
        filho.chaves.append(no.chaves.pop(i))

        # Puxa o conteúdo do irmão da direita
        filho.chaves.extend(irmao.chaves)
        if not filho.leaf:
            filho.filhos.extend(irmao.filhos)

        # Deleta o irmão da direita
        no.filhos.pop(i + 1)

In [9]:
if __name__ == "__main__":
    print("=== INICIANDO TESTES DA ÁRVORE B ===\n")
    
    # Criando uma árvore com grau mínimo t=2
    # Isso significa que cada nó terá no máximo 3 chaves e no mínimo 1.
    arvore = ArvoreB(t=2)

    # 1. TESTE DE INSERÇÃO
    print("1. Inserindo anotações fora de ordem...")
    insercoes = [
        Anotacao(10, 1, "Introdução ao assunto"),
        Anotacao(5,  2, "Definição de termos"),
        Anotacao(20, 1, "Conclusão"),
        Anotacao(15, 3, "Exemplos práticos"),
        Anotacao(5,  1, "Prefácio"),           # Empate de página, desempata no parágrafo
        Anotacao(30, 1, "Referências"),
        Anotacao(25, 2, "Bibliografia")
    ]

    for anotacao in insercoes:
        arvore.inserir(anotacao)
        print(f"Inserida: {anotacao}")

    print("\n--- Listagem após inserções (Deve estar ordenada) ---")
    print(arvore.listar_anotacoes())


    # 2. TESTE DE CONSULTA
    print("\n\n2. Realizando consultas...")
    
    # Buscando uma que existe
    resultado_sucesso = arvore.consultar(15, 3)
    if resultado_sucesso:
        print(f"Sucesso! Encontrei: {resultado_sucesso} -> Texto: '{resultado_sucesso.texto}'")
    else:
        print("Erro: Não encontrou (15, 3).")

    # Buscando uma que NÃO existe
    resultado_falha = arvore.consultar(100, 1)
    if resultado_falha is None:
        print("Sucesso! O sistema retornou None para (100, 1), pois não existe.")
    else:
        print("Erro: Retornou algo que não deveria existir.")


    # 3. TESTE DE REMOÇÃO
    print("\n\n3. Removendo anotações...")
    
    print("Removendo (5, 2)...")
    arvore.remover(5, 2)
    
    print("Removendo (25, 2)...")
    arvore.remover(25, 2)

    print("Removendo (10, 1)...") # Removendo uma chave que provavelmente está no meio da árvore
    arvore.remover(10, 1)

    print("\n--- Listagem após remoções ---")
    print(arvore.listar_anotacoes())
    
    print("\n=== FIM DOS TESTES ===")

=== INICIANDO TESTES DA ÁRVORE B ===

1. Inserindo anotações fora de ordem...
Inserida: Anotacao(Pg:10, Par:1)
Inserida: Anotacao(Pg:5, Par:2)
Inserida: Anotacao(Pg:20, Par:1)
Inserida: Anotacao(Pg:15, Par:3)
Inserida: Anotacao(Pg:5, Par:1)
Inserida: Anotacao(Pg:30, Par:1)
Inserida: Anotacao(Pg:25, Par:2)

--- Listagem após inserções (Deve estar ordenada) ---
[Anotacao(Pg:5, Par:1), Anotacao(Pg:5, Par:2), Anotacao(Pg:10, Par:1), Anotacao(Pg:15, Par:3), Anotacao(Pg:20, Par:1), Anotacao(Pg:25, Par:2), Anotacao(Pg:30, Par:1)]


2. Realizando consultas...
Sucesso! Encontrei: Anotacao(Pg:15, Par:3) -> Texto: 'Exemplos práticos'
Sucesso! O sistema retornou None para (100, 1), pois não existe.


3. Removendo anotações...
Removendo (5, 2)...
Removendo (25, 2)...
Removendo (10, 1)...

--- Listagem após remoções ---
[Anotacao(Pg:5, Par:1), Anotacao(Pg:15, Par:3), Anotacao(Pg:20, Par:1), Anotacao(Pg:30, Par:1)]

=== FIM DOS TESTES ===
